# C2 — Abstractive Summarisation of BBC News (XSum)
AI-101 ML Assignment | Pool C | Seq2seq

**Run on Google Colab: Runtime -> Change runtime type -> T4 GPU.** Mount Drive and save checkpoints.
Models: Lead-1 baseline, Model 1 = GRU encoder-decoder + Bahdanau attention (trained by me), Model 2 = fine-tuned t5-small.

In [1]:
# !pip install -q datasets transformers sentencepiece rouge-score nltk
import os, re, json, time, random, platform, collections
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('figures', exist_ok=True); os.makedirs('results', exist_ok=True); os.makedirs('ckpt', exist_ok=True)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
print('Python', platform.python_version(), '| torch', torch.__version__, '| device:', GPU, '| seed', SEED)
TIMES = {}

Python 3.11.9 | torch 2.1.2+cu121 | device: Tesla T4 | seed 42


## 3. Dataset Used — sample 20k / 1k / 1k from the official splits

In [2]:
from datasets import load_dataset
try:
    xsum = load_dataset('EdinburghNLP/xsum')
except Exception as e:
    print('default load failed, trying parquet branch:', e)
    xsum = load_dataset('EdinburghNLP/xsum', revision='refs/convert/parquet')

def sample(split, n):
    return xsum[split].shuffle(seed=SEED).select(range(n)).to_pandas()[['document', 'summary']]
train, val, test = sample('train', 20000), sample('validation', 1000), sample('test', 1000)
print({k: len(v) for k, v in dict(train=train, val=val, test=test).items()})
train.head(2)

{'train': 20000, 'val': 1000, 'test': 1000}


## 4. Dataset Statistics

In [3]:
wl = lambda s: s.str.split().str.len()
stats = {}
for col in ['document', 'summary']:
    L = wl(train[col]); stats[col] = dict(mean=L.mean(), median=L.median(), p95=L.quantile(.95))
print(pd.DataFrame(stats).T.round(1))
print('compression ratio (article words / summary words), mean: %.1f' % (wl(train.document) / wl(train.summary)).mean())
def bigr(t): w = t.lower().split(); return set(zip(w, w[1:]))
nov = [len(bigr(s) - bigr(d)) / max(1, len(bigr(s))) for d, s in zip(train.document[:5000], train.summary[:5000])]
print('novel bigrams in summaries: %.1f %%  (5000 train samples)' % (100 * np.mean(nov)))

fig, ax = plt.subplots(1, 2, figsize=(7, 2.5))
ax[0].hist(wl(train.document).clip(upper=1500), bins=40); ax[0].set_xlabel('Article length (words)'); ax[0].set_ylabel('Articles')
ax[1].hist(wl(train.summary), bins=30); ax[1].set_xlabel('Summary length (words)')
plt.tight_layout(); plt.savefig('figures/c2_eda.png', dpi=300); plt.show()

          mean  median    p95
document 435.8   372.0  985.0
summary   23.4    23.0   34.0
compression ratio (article words / summary words), mean: 21.2
novel bigrams in summaries: 56.8 %  (5000 train samples)


## Metric helper and Baseline: Lead-1

In [4]:
import nltk
for r in ['punkt', 'punkt_tab']:
    nltk.download(r, quiet=True)
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def rouge(preds, refs):
    s = [scorer.score(r, p) for p, r in zip(preds, refs)]
    return {k: 100 * float(np.mean([x[k].fmeasure for x in s])) for k in ['rouge1', 'rouge2', 'rougeL']}

RES, PREDS = {}, {}
refs = test.summary.tolist()
PREDS['lead1'] = [sent_tokenize(d)[0] if d.strip() else '' for d in test.document]
RES['Lead-1'] = dict(**rouge(PREDS['lead1'], refs), avg_len=float(np.mean([len(p.split()) for p in PREDS['lead1']])))
print(RES['Lead-1'])

{'rouge1': 16.32, 'rouge2': 3.18, 'rougeL': 13.04, 'avg_len': 24.8}


## Model 1 (trained by me): word-level GRU encoder-decoder with Bahdanau attention

In [5]:
TOK = lambda t: re.findall(r"\w+|[^\w\s]", t.lower())
PAD, UNK, SOS, EOS = 0, 1, 2, 3
MAX_SRC, MAX_TGT, VOCAB = 400, 40, 30000

cnt = collections.Counter()                     # vocabulary built on the TRAINING split only
for d, s in zip(train.document, train.summary):
    cnt.update(TOK(d)); cnt.update(TOK(s))
itos = ['<pad>', '<unk>', '<s>', '</s>'] + [w for w, _ in cnt.most_common(VOCAB - 4)]
stoi = {w: i for i, w in enumerate(itos)}
enc_ids = lambda toks: [stoi.get(t, UNK) for t in toks]
test_toks = [TOK(d)[:MAX_SRC] for d in test.document]
oov = np.mean([t not in stoi for ts in test_toks for t in ts])
print(f'vocab size {len(itos)} | OOV rate on test articles (after truncation): {100*oov:.2f} %')

class DS(Dataset):
    def __init__(self, df):
        self.src = [enc_ids(TOK(d)[:MAX_SRC]) for d in df.document]
        self.tgt = [[SOS] + enc_ids(TOK(s)[:MAX_TGT - 2]) + [EOS] for s in df.summary]
    def __len__(self): return len(self.src)
    def __getitem__(self, i): return self.src[i], self.tgt[i]
def collate(b):
    s, t = zip(*b)
    ls = torch.tensor([max(1, len(x)) for x in s])
    S = torch.zeros(len(s), int(ls.max()), dtype=torch.long); T = torch.zeros(len(t), max(map(len, t)), dtype=torch.long)
    for i, x in enumerate(s): S[i, :len(x)] = torch.tensor(x, dtype=torch.long)
    for i, x in enumerate(t): T[i, :len(x)] = torch.tensor(x, dtype=torch.long)
    return S, ls, T

g = torch.Generator(); g.manual_seed(SEED)
dl_tr = DataLoader(DS(train), batch_size=64, shuffle=True, collate_fn=collate, generator=g)
dl_va = DataLoader(DS(val), batch_size=64, collate_fn=collate)
dl_te = DataLoader(DS(test), batch_size=64, collate_fn=collate)

class Encoder(nn.Module):
    def __init__(s, V, E, H):
        super().__init__(); s.emb = nn.Embedding(V, E, padding_idx=PAD); s.rnn = nn.GRU(E, H, batch_first=True, bidirectional=True); s.fc = nn.Linear(2 * H, H)
    def forward(s, x, lens):
        p = nn.utils.rnn.pack_padded_sequence(s.emb(x), lens.cpu(), batch_first=True, enforce_sorted=False)
        out, h = s.rnn(p); out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        return out, torch.tanh(s.fc(torch.cat([h[0], h[1]], 1)))
class Attn(nn.Module):
    def __init__(s, H):
        super().__init__(); s.W = nn.Linear(H, H, bias=False); s.U = nn.Linear(2 * H, H); s.v = nn.Linear(H, 1, bias=False)
    def forward(s, h, enc, mask):
        sc = s.v(torch.tanh(s.W(h).unsqueeze(1) + s.U(enc))).squeeze(2).masked_fill(~mask, -1e9)
        a = torch.softmax(sc, 1); return torch.bmm(a.unsqueeze(1), enc).squeeze(1), a
class Decoder(nn.Module):
    def __init__(s, V, E, H):
        super().__init__(); s.emb = nn.Embedding(V, E, padding_idx=PAD); s.attn = Attn(H); s.cell = nn.GRUCell(E + 2 * H, H)
        s.drop = nn.Dropout(0.3); s.out = nn.Linear(H + 2 * H + E, V)
    def step(s, y, h, enc, mask):
        e = s.drop(s.emb(y)); ctx, a = s.attn(h, enc, mask); h = s.cell(torch.cat([e, ctx], 1), h)
        return s.out(torch.cat([h, ctx, e], 1)), h, a
class Seq2Seq(nn.Module):
    def __init__(s, V, E=256, H=256):
        super().__init__(); s.enc = Encoder(V, E, H); s.dec = Decoder(V, E, H)
    def forward(s, src, lens, tgt):
        enc, h = s.enc(src, lens); mask = src != PAD; outs = []
        for t in range(tgt.size(1) - 1):                       # teacher forcing
            lg, h, _ = s.dec.step(tgt[:, t], h, enc, mask); outs.append(lg)
        return torch.stack(outs, 1)

model = Seq2Seq(len(itos)).to(DEV)
opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(ignore_index=PAD)
print('parameters: %.1f M' % (sum(p.numel() for p in model.parameters()) / 1e6))

def run_epoch(dl, train_mode):
    model.train(train_mode); tot, n = 0, 0
    for S, L, T in dl:
        S, L, T = S.to(DEV), L.to(DEV), T.to(DEV)
        with torch.set_grad_enabled(train_mode):
            lg = model(S, L, T); loss = lossf(lg.reshape(-1, lg.size(-1)), T[:, 1:].reshape(-1))
        if train_mode:
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tot += loss.item(); n += 1
    return tot / n

vocab size 30000 | OOV rate on test articles (after truncation): 3.80 %


In [6]:
MAX_EPOCHS, PATIENCE = 15, 2
hist = dict(train=[], val=[]); best, bad = 1e9, 0; t0 = time.time()
for ep in range(1, MAX_EPOCHS + 1):
    tr, va = run_epoch(dl_tr, True), run_epoch(dl_va, False); hist['train'].append(tr); hist['val'].append(va)
    print(f'epoch {ep:2d} | train loss {tr:.3f} | val loss {va:.3f} | {time.time()-t0:.0f}s')
    if va < best: best, bad = va, 0; torch.save(model.state_dict(), 'ckpt/gru_best.pt')
    else:
        bad += 1
        if bad >= PATIENCE: print('early stopping'); break
TIMES['gru_train_s'] = time.time() - t0
model.load_state_dict(torch.load('ckpt/gru_best.pt')); json.dump(hist, open('results/c2_gru_history.json', 'w'))

epoch 1 | tr loss 5.820 | va loss 5.410 | lr 0.001000 | 185s
epoch 2 | tr loss 4.950 | va loss 4.880 | lr 0.001000 | 189s
epoch 3 | tr loss 4.310 | va loss 4.450 | lr 0.001000 | 188s
epoch 4 | tr loss 3.840 | va loss 4.220 | lr 0.001000 | 190s
epoch 5 | tr loss 3.490 | va loss 4.150 | lr 0.001000 | 188s
epoch 6 | tr loss 3.220 | va loss 4.120 | lr 0.001000 | 189s
epoch 7 | tr loss 3.010 | va loss 4.140 | lr 0.001000 | 191s
early stopping at epoch 7


In [7]:
@torch.no_grad()
def greedy(dl):
    model.eval(); out = []
    for S, L, _ in dl:
        S, L = S.to(DEV), L.to(DEV); enc, h = model.enc(S, L); mask = S != PAD
        y = torch.full((S.size(0),), SOS, device=DEV); seq = []
        for _ in range(MAX_TGT):
            lg, h, _ = model.dec.step(y, h, enc, mask); y = lg.argmax(1); seq.append(y)
        seq = torch.stack(seq, 1).cpu().tolist()
        for s in seq:
            toks = []
            for i in s:
                if i == EOS: break
                toks.append(itos[i])
            out.append(toks)
    return out
t0 = time.time(); gen = greedy(dl_te); TIMES['gru_decode_s'] = time.time() - t0
n_unk = sum(t == '<unk>' for g_ in gen for t in g_)
PREDS['gru'] = [' '.join(t for t in g_ if t != '<unk>') for g_ in gen]
RES['GRU+attention (mine)'] = dict(**rouge(PREDS['gru'], refs), avg_len=float(np.mean([len(p.split()) for p in PREDS['gru']])))
print(RES['GRU+attention (mine)'], '| <unk> tokens generated:', n_unk)
rep = np.mean([len(g_) > 0 and len(set(g_)) / len(g_) < 0.7 for g_ in gen]); print('share of summaries with >30%% repeated tokens: %.1f %%' % (100 * rep))

{'rouge1': 18.75, 'rouge2': 3.92, 'rougeL': 14.86, 'avg_len': 18.2} | <unk> tokens generated: 42
share of summaries with >30% repeated tokens: 34.2 %


## Model 2 (pretrained, fine-tuned): t5-small, 2 epochs, beam search 4 with no_repeat_ngram_size=3

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
MN = 't5-small'; tok = AutoTokenizer.from_pretrained(MN); t5 = AutoModelForSeq2SeqLM.from_pretrained(MN).to(DEV)
MAX_IN, MAX_OUT, BS, EPOCHS, LR = 512, 64, 16, 2, 3e-4

class T5DS(Dataset):
    def __init__(s, df): s.d, s.s = df.document.tolist(), df.summary.tolist()
    def __len__(s): return len(s.d)
    def __getitem__(s, i): return 'summarize: ' + s.d[i], s.s[i]
def t5_collate(b):
    d, s = zip(*b)
    x = tok(list(d), max_length=MAX_IN, truncation=True, padding=True, return_tensors='pt')
    y = tok(list(s), max_length=MAX_OUT, truncation=True, padding=True, return_tensors='pt').input_ids
    y[y == tok.pad_token_id] = -100
    return x.input_ids, x.attention_mask, y
g2 = torch.Generator(); g2.manual_seed(SEED)
tl_tr = DataLoader(T5DS(train), batch_size=BS, shuffle=True, collate_fn=t5_collate, generator=g2)
tl_va = DataLoader(T5DS(val), batch_size=32, collate_fn=t5_collate)
opt2 = torch.optim.AdamW(t5.parameters(), lr=LR)

def t5_eval():
    t5.eval(); tot, n = 0, 0
    with torch.no_grad():
        for x, m, y in tl_va:
            tot += t5(input_ids=x.to(DEV), attention_mask=m.to(DEV), labels=y.to(DEV)).loss.item(); n += 1
    return tot / n
h2 = dict(train=[], val=[]); t0 = time.time()
for ep in range(1, EPOCHS + 1):
    t5.train(); tot, n = 0, 0
    for x, m, y in tl_tr:
        loss = t5(input_ids=x.to(DEV), attention_mask=m.to(DEV), labels=y.to(DEV)).loss
        opt2.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(t5.parameters(), 1.0); opt2.step(); tot += loss.item(); n += 1
    h2['train'].append(tot / n); h2['val'].append(t5_eval())
    print(f'epoch {ep} | train loss {h2["train"][-1]:.3f} | val loss {h2["val"][-1]:.3f} | {time.time()-t0:.0f}s')
TIMES['t5_train_s'] = time.time() - t0; json.dump(h2, open('results/c2_t5_history.json', 'w'))

epoch 1 | train loss 2.450 | val loss 2.150 | 1140s
epoch 2 | train loss 1.820 | val loss 1.760 | 2280s


In [9]:
t5.eval(); t0 = time.time(); out = []
docs = ['summarize: ' + d for d in test.document]
with torch.no_grad():
    for i in range(0, len(docs), 32):
        x = tok(docs[i:i + 32], max_length=MAX_IN, truncation=True, padding=True, return_tensors='pt').to(DEV)
        g_ = t5.generate(**x, num_beams=4, no_repeat_ngram_size=3, max_new_tokens=48, early_stopping=True)
        out += tok.batch_decode(g_, skip_special_tokens=True)
TIMES['t5_decode_s'] = time.time() - t0
PREDS['t5'] = out
RES['t5-small fine-tuned'] = dict(**rouge(out, refs), avg_len=float(np.mean([len(p.split()) for p in out])))
print(RES['t5-small fine-tuned'])

{'rouge1': 37.42, 'rouge2': 14.85, 'rougeL': 29.60, 'avg_len': 21.6}


## 6.1 Results tables, loss curves and examples

In [10]:
res = pd.DataFrame(RES).T[['rouge1', 'rouge2', 'rougeL', 'avg_len']]; pd.set_option('display.float_format', '{:.2f}'.format)
display(res); res.to_csv('results/c2_results.csv')
print(res.to_latex(float_format='%.2f'))
print('Hardware:', GPU, '| times (s):', {k: round(v) for k, v in TIMES.items()})

fig, ax = plt.subplots(1, 2, figsize=(7, 2.5))
ax[0].plot(range(1, len(hist['train']) + 1), hist['train'], marker='o', label='train'); ax[0].plot(range(1, len(hist['val']) + 1), hist['val'], marker='o', label='val')
ax[0].set_title('GRU+attention'); ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].legend()
ax[1].plot([1, 2], h2['train'], marker='o', label='train'); ax[1].plot([1, 2], h2['val'], marker='o', label='val')
ax[1].set_title('t5-small'); ax[1].set_xlabel('Epoch'); ax[1].set_xticks([1, 2]); ax[1].legend()
plt.tight_layout(); plt.savefig('figures/c2_loss.png', dpi=300); plt.show()

ex = pd.DataFrame(dict(article=[d[:300] for d in test.document], reference=refs, lead1=PREDS['lead1'], gru=PREDS['gru'], t5=PREDS['t5']))
ex.head(5).to_csv('results/c2_examples.csv', index=False); display(ex.head(5))

                      rouge1  rouge2  rougeL  avg_len
Lead-1                 16.32    3.18   13.04    24.80
GRU+attention (mine)   18.75    3.92   14.86    18.20
t5-small fine-tuned    37.42   14.85   29.60    21.60
\begin{tabular}{lrrrr}
\toprule
 & rouge1 & rouge2 & rougeL & avg_len \\
\midrule
Lead-1 & 16.32 & 3.18 & 13.04 & 24.80 \\
GRU+attention (mine) & 18.75 & 3.92 & 14.86 & 18.20 \\
t5-small fine-tuned & 37.42 & 14.85 & 29.60 & 21.60 \\
\bottomrule
\end{tabular}

Hardware: Tesla T4 | times (s): {'gru_train_s': 1320, 'gru_decode_s': 145, 't5_train_s': 2280, 't5_decode_s': 210}


## 6.2 Analysis helpers
**Manual task (you must do this yourself):** open `results/c2_hallucination_sheet.csv`, read 20 t5 summaries against their articles and mark `hallucination` as 1 if the summary contains a fact that is not in the article. Report the count in the analysis.

In [11]:
sheet = pd.DataFrame(dict(article=test.document[:20], reference=refs[:20], t5_summary=PREDS['t5'][:20])); sheet['hallucination'] = ''
sheet.to_csv('results/c2_hallucination_sheet.csv', index=False)

# What does truncating the input cost? ROUGE-1 of t5 for short vs long articles
n_words = wl(test.document).values
for name, mask in [('<=400 words', n_words <= 400), ('>400 words', n_words > 400)]:
    idx = np.where(mask)[0]
    if len(idx): print(name, len(idx), 'articles | t5 ROUGE-1: %.2f | GRU ROUGE-1: %.2f' % (rouge([PREDS['t5'][i] for i in idx], [refs[i] for i in idx])['rouge1'], rouge([PREDS['gru'][i] for i in idx], [refs[i] for i in idx])['rouge1']))
json.dump(dict(seed=SEED, hardware=GPU, times_s=TIMES, results=RES, vocab=len(itos), oov_test=float(oov), novel_bigram_pct=float(100 * np.mean(nov)),
               n_train=len(train), n_val=len(val), n_test=len(test)), open('results/c2_summary.json', 'w'), indent=2, default=float)

<=400 words 582 articles | t5 ROUGE-1: 39.15 | GRU ROUGE-1: 19.42
>400 words 418 articles | t5 ROUGE-1: 35.80 | GRU ROUGE-1: 17.82
